<a href="https://colab.research.google.com/github/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/blob/main/Lecture-13_March-5-2026/Lecture-13_Regression-with-LightGBM_Solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Lecture 13 - Regression

Here we are going to regression modeling on the Bradley Melting Point Dataset, which is curated chemical dataset with melting points of around 3,000 chemical compounds, see [here](https://www.kaggle.com/datasets/aliffaagnur/melting-point-chemical-dataset/data).

We will use the [LightGBM](https://en.wikipedia.org/wiki/LightGBM) method is seperate from scikit-learn but uses the same API.

This is heavily inspired by this tutorial:
- [Building a Simple Regression Model](https://colab.research.google.com/github/PatWalters/practical_cheminformatics_tutorials/blob/main/ml_models/regression_model.ipynb)


Install RDKit and [LightGBM](https://lightgbm.readthedocs.io/en/stable/)

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install rdkit lightgbm mols2grid

Import all basic pacakges

In [ ]:
# basic
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# For progress bar
from tqdm.auto import tqdm

# RDkit
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

# scikit-learn
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestRegressor

# LightGBM
from lightgbm import LGBMRegressor, plot_importance

# mols2grid
import mols2grid

# To visluzise results
from yellowbrick.regressor import prediction_error, ResidualsPlot


Download dataset

In [ ]:
# Bash script to download all the dataset. Don't worry if you don't understand it
%%bash

url="https://raw.githubusercontent.com/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/refs/heads/main/Assignment-2/"
dataset_filename="BradleyDoublePlusGoodMeltingPointDataset.csv"

rm -f ${dataset_filename}

wget ${url}/${dataset_filename} &> /dev/null

ls

This is to add progress bars panda task

In [ ]:
tqdm.pandas()

Read dataset

In [ ]:
data_mp = pd.read_csv("BradleyDoublePlusGoodMeltingPointDataset.csv")



In [ ]:
data_mp

In [ ]:
# Simplify by removing columns from the data frame
data_mp = data_mp.drop(columns=['csid','link','source','count','min','max','range'])

Let's visualize the molecules using mols2grid.

In [ ]:
def mp_str(x):
    return f'{x:.2f} C'

mols2grid.display(data_mp,
                  smiles_col='smiles',
                  subset=['img','name','mpC'],
                  transform={"mpC": mp_str})

## SMILES ARbitrary Target Specification (SMARTS)

SMARTS is a very useful method to look for substructures in molecules using RDKit and it can be used in various ways in RDKit, for example to delete parts. We can use it to search in the mols2grid above.

See these excellent tutorials for further information about SMARTS:
- [An introduction to the SMILES ARbitrary Target Specification (SMARTS)](https://colab.research.google.com/github/PatWalters/practical_cheminformatics_tutorials/blob/main/fundamentals/SMARTS_tutorial.ipynb)
- [Recursive SMARTS](https://colab.research.google.com/github/PatWalters/practical_cheminformatics_tutorials/blob/main/fundamentals/recursive_smarts.ipynb)
- [SMARTS Tutorial](https://www.daylight.com/dayhtml_tutorials/languages/smarts/)

For example, we can use the following:
- Benzene rings: `c1ccccc1`
- Atoms with charges -1 or +1: `[-1]` or `[+1]`
- Nitrogen or oxygen attached to an aliphatic carbon: `C[#7,#8]`

Try to search for certain SMARTS string in the mols2grid object above.

## Calculate RDKit Features

Here we use a quick way to calculate a large set of properties. Will be explained in class.

In [ ]:
property_names = list(rdMolDescriptors.Properties.GetAvailableProperties())

In [ ]:
property_names

In [ ]:
property_getter = rdMolDescriptors.Properties(property_names)

In [ ]:
property_getter

In [ ]:
mol = Chem.MolFromSmiles("c1ccccc1CC")
print(np.array(property_getter.ComputeProperties(mol)))

In [ ]:
property_names_2 = ['tpsa', 'NumRotatableBonds']
property_getter_2 = rdMolDescriptors.Properties(property_names_2)
mol = Chem.MolFromSmiles("c1ccccc1CC")
print(np.array(property_getter_2.ComputeProperties(mol)))

In [ ]:
def smi2props(smi):
    mol = Chem.MolFromSmiles(smi)
    props = None
    if mol:
        props = np.array(property_getter.ComputeProperties(mol))
    return props

In [ ]:
# one to the calculations for all smiles strings
data_mp['props'] = [smi2props(s) for s in data_mp['smiles'] ]

In [ ]:
data_mp['props']

In [ ]:
data_mp.mpC

In [ ]:
# These two ways are equilivant

data_mp['props2'] = data_mp.smiles.progress_apply(smi2props)

# data_mp['props'] = data_mp['smiles'].progress_apply(smi2props)

In [ ]:
data_mp[['props','props2']]

In [ ]:
data_mp = data_mp.drop(columns=['props','props2'])

In [ ]:
data_mp

In [ ]:
data_mp['props'] = data_mp.smiles.progress_apply(smi2props)

In [ ]:
data_mp

In [ ]:
data_mp = data_mp.dropna(subset=['props'])

In [ ]:
data_mp[property_names] = data_mp['props'].tolist()

In [ ]:
data_mp

In [ ]:
data_mp.drop("props",axis=1,inplace=True)

### Visualize Features and Target

In [ ]:
plt.hist(data_mp.mpC,bins=60,density=True,label='Discrete Histogram')
sns.kdeplot(data_mp.mpC,label='KDE')
plt.title("mpC")
plt.xlabel("mpC")
plt.show()

for f in property_names:
  plt.hist(data_mp[f],bins=60,density=True,label='Discrete Histogram')
  sns.kdeplot(data_mp[f],label='KDE')
  plt.title(f)
  plt.xlabel(f)
  plt.show()

In [ ]:
print("Number of features: ",len(property_names))

## Train using LightGBM

In [ ]:
train, test = train_test_split(data_mp,test_size=0.20)

In [ ]:
train_X = train[property_names]
train_y = train.mpC
test_X = test[property_names]
test_y = test.mpC

In [ ]:
lgbm = LGBMRegressor()
lgbm.fit(train_X, train_y)

In [ ]:
test_pred = lgbm.predict(test_X)
train_pred = lgbm.predict(train_X)

In [ ]:
plt.plot(test_y, test_pred,'.',label='Test set')
plt.plot(train_y, train_pred,'.',label='Train set')
# to get a diagonal line
diagonal_line =np.linspace(np.min(train_y),np.max(train_y),1000)
plt.plot(diagonal_line,diagonal_line, color='red', linestyle='--')
# plt.axline([0, 0], slope=1, color='red', linestyle='--')
plt.legend()
plt.ylabel("Predicted Value")
plt.xlabel("Actual Value")
plt.show()

In [ ]:
ax = sns.scatterplot(x=test_y,y=test_pred)
ax.set(xlabel="Experimental mpC")
ax.set(ylabel="Predicted mpC")

In [ ]:
ax = sns.regplot(x=test_y,y=pred,scatter_kws={'s':10})
ax.set(xlabel="Experimental mpC")
ax.set(ylabel="Predicted mpC")

In [ ]:
del pred

In [ ]:
metrics.r2_score(test_y,test_pred)

In [ ]:
metrics.root_mean_squared_error(test_y,test_pred)

[Yellowbrick](https://www.scikit-yb.org/en/latest/) is python package that extends scikit-learn in various ways, including visualization, that we use here.

In [ ]:
visualizer = prediction_error(lgbm, train_X, train_y, test_X, test_y,alpha=0.35)

In [ ]:
visualizer = ResidualsPlot(lgbm)
visualizer.fit(train_X, train_y)
visualizer.score(test_X, test_y)
visualizer.show();

### Feature Importance

Let's do feature importance

In [ ]:
result = permutation_importance(
    lgbm, test_X, test_y, n_repeats=10, random_state=42, n_jobs=2
)

feature_importances = pd.Series(result.importances_mean, index=property_names)

fig, ax = plt.subplots()
feature_importances.plot.bar(yerr=result.importances_std, ax=ax)
ax.set_title("Feature importances using permutation on full model")
ax.set_ylabel("Mean accuracy decrease")
fig.tight_layout()
plt.show()

In [ ]:
# LightGBM has an built-in function to look at feature importance
plot_importance(lgbm,figsize=(8,12),height=0.5)

### Log10 Transform



In [ ]:
data_mp_postive = data_mp[data_mp.mpC > 0]


In [ ]:
data_mp_postive

In [ ]:
train, test = train_test_split(data_mp_postive,test_size=0.20)
train_X = train[property_names]
train_y = train.mpC_log10
test_X = test[property_names]
test_y = test.mpC_log10

In [ ]:
lgbm = LGBMRegressor()
lgbm.fit(train_X, train_y)
test_pred = lgbm.predict(test_X)
train_pred = lgbm.predict(train_X)

In [ ]:
plt.plot(test_y, test_pred,'.',label='Test set')
plt.plot(train_y, train_pred,'.',label='Train set')
# to get a diagonal line
diagonal_line =np.linspace(np.min(train_y),np.max(train_y),1000)
plt.plot(diagonal_line,diagonal_line, color='red', linestyle='--')
# plt.axline([0, 0], slope=1, color='red', linestyle='--')
plt.legend()
plt.ylabel("Predicted Value")
plt.xlabel("Actual Value")
plt.show()

In [ ]:
plt.plot(10**test_y, 10**test_pred,'.',label='Test set')
plt.plot(10**train_y, 10**train_pred,'.',label='Train set')
# to get a diagonal line
diagonal_line =np.linspace(np.min(10**train_y),np.max(10**train_y),1000)
plt.plot(diagonal_line,diagonal_line, color='red', linestyle='--')
# plt.axline([0, 0], slope=1, color='red', linestyle='--')
plt.legend()
plt.ylabel("Predicted Value")
plt.xlabel("Actual Value")
plt.show()

### Scramble



In [ ]:
train, test = train_test_split(data_mp,test_size=0.20)
train_X = train[property_names]
train_y = train.mpC
test_X = test[property_names]
test_y = test.mpC

In [ ]:
scrambled=True
rng = np.random.RandomState(100)
train_y_used = train_y.sample(frac=1.0, random_state=rng).reset_index(drop=True) if scrambled else train_y.copy()

lgbm = LGBMRegressor()
lgbm.fit(train_X, train_y_used)
test_pred = lgbm.predict(test_X)
train_pred = lgbm.predict(train_X)

In [ ]:
plt.plot(test_y, test_pred,'.',label='Test set')
plt.plot(train_y, train_pred,'.',label='Train set')
# to get a diagonal line
diagonal_line =np.linspace(np.min(train_y),np.max(train_y),1000)
plt.plot(diagonal_line,diagonal_line, color='red', linestyle='--')
# plt.axline([0, 0], slope=1, color='red', linestyle='--')
plt.legend()
plt.ylabel("Predicted Value")
plt.xlabel("Actual Value")
plt.show()

## Cross Validation

In [ ]:
r2_list = []
rmse_list = []
NumberOfRandomShuffles=100
for i in tqdm(range(0,NumberOfRandomShuffles)):
    # setup training and test sets
    train, test = train_test_split(data_mp)
    train_X = train[property_names]
    train_y = train.mpC
    test_X = test[property_names]
    test_y = test.mpC
    # create the regressor
    lgbm = LGBMRegressor()
    # train the model
    lgbm.fit(train_X,train_y)
    pred = lgbm.predict(test_X)
    r2 = metrics.r2_score(test_y,pred)
    rmse = metrics.root_mean_squared_error(test_y,pred)
    print(r2,rmse)
    r2_list.append(r2)
    rmse_list.append(rmse)

In [ ]:
## Calculate mean and stddev
r2_mean = np.mean(r2_list)
r2_stddev = np.std(r2_list)

rmse_mean = np.mean(rmse_list)
rmse_stddev = np.std(rmse_list)


print("R^2:  {:.3f} +- {:.3f}".format(r2_mean,r2_stddev))
print("RMSE: {:.3f} +- {:.3f}".format(rmse_mean,rmse_stddev))

In [ ]:
ax = sns.boxplot(x=r2_list)
ax.set(xlim=(0,1))
ax.set(xlabel="R$^2$")

In [ ]:
ax = sns.boxplot(x=rmse_list)
ax.set(xlabel="RMSE")

### Random Forrest

Repeat the analysis using Random Forrest Regreesion